# CTMC Journey Models: Success vs 60-Day Inactivity Failure

This notebook compares the retained CTMC baselines against two timeout-aware models. The key target distinction is:

- old CTMC target: `P(hit order_shipped state 28 within a horizon | current state)`
- classification target: `P(success before failure by 60 days of customer inactivity | current state)`

Company/system events are not treated as customer actions for the inactivity clock.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src" / "models"))
sys.path.insert(0, str(PROJECT_ROOT / "src" / "visualizations"))

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score

from ctmc import (
    CTMCData,
    ClusteredCTMC,
    GlobalCTMC,
    SemiMarkovTimeoutModel,
    TimeoutAbsorbingCTMC,
    FAILURE_STATE,
    SUCCESS_STATE,
)
from ctmc_plots import plot_generator_heatmap, plot_top_transition_graph

MAX_JOURNEYS = 100_000
N_CLUSTERS = 3
USE_SPECTRAL_CLUSTERING = False
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 80)

## Event Definitions and State Choice

The event definition file mixes customer actions with company/system states. For inactivity failure, company/system events should not reset the 60-day clock. The models below use customer-action states plus terminal `order_shipped=28` where needed.

In [ ]:
data = CTMCData()
event_defs = pd.read_csv(PROJECT_ROOT / "data" / "Event Definitions.csv")
customer_states = data.customer_action_states(include_success=False)

event_defs["used_as_customer_action_state"] = event_defs["event_definition_id"].isin(customer_states)
display(event_defs.sort_values("event_definition_id"))
print("Customer action states:", sorted(customer_states))
print("Success terminal state:", SUCCESS_STATE, "Failure terminal state:", FAILURE_STATE)

## Build Transition Tables

`transitions` is the old CTMC transition table, but filtered to customer-action states plus success. `timeout_transitions` adds an absorbing failure state for incomplete journeys after 60 days with no customer action.

In [ ]:
transitions = data.transition_table(
    max_journeys=MAX_JOURNEYS,
    customer_actions_only=True,
    include_success=True,
)
timeout_transitions = data.timeout_transition_table(max_journeys=MAX_JOURNEYS)

print("customer-action CTMC transitions:", transitions.shape, "journeys:", transitions["id"].nunique())
print("timeout-aware transitions:", timeout_transitions.shape, "journeys:", timeout_transitions["id"].nunique())
display(transitions.head())
display(timeout_transitions.tail())

## Old Baseline 1: Global CTMC

This still computes `P(hit order_shipped within 60 days | current state)`. It is useful, but it is not the exact failure-vs-success target.

In [ ]:
global_ctmc = GlobalCTMC().fit(transitions)
print("Q shape:", global_ctmc.Q_.shape)
display(global_ctmc.top_rates(20))

plot_generator_heatmap(global_ctmc, RESULTS_DIR / "ctmc_customer_action_generator_heatmap.png")
plot_top_transition_graph(global_ctmc, RESULTS_DIR / "ctmc_customer_action_top_transition_graph.png")

## Old Baseline 2: Clustered CTMC

This is the retained segmented CTMC baseline. Set `USE_SPECTRAL_CLUSTERING = True` in the first code cell to compare spectral clustering against KMeans.

In [ ]:
clustered = ClusteredCTMC(
    n_clusters=N_CLUSTERS,
    random_state=42,
    use_spectral_clustering=USE_SPECTRAL_CLUSTERING,
).fit(transitions)
cluster_summary = clustered.cluster_summary()
display(cluster_summary)

cluster_summary.plot.bar(x="cluster", y="n_journeys", legend=False, figsize=(6, 4))
plt.title("Journey clusters")
plt.ylabel("n journeys")
plt.tight_layout()
plt.show()

features = clustered.feature_builder.transform(transitions)

## New Model 1: Timeout-Absorbing CTMC

This augments the CTMC state space with an absorbing failure state. It estimates competing rates to success and failure, then solves for `P(absorb in success before failure)`.

In [ ]:
timeout_ctmc = TimeoutAbsorbingCTMC().fit(timeout_transitions)
timeout_probs_by_state = pd.DataFrame({
    "state": timeout_ctmc.states_,
    "p_success_before_timeout_failure": timeout_ctmc.predict_success_probability(timeout_ctmc.states_),
}).sort_values("p_success_before_timeout_failure", ascending=False)
display(timeout_probs_by_state)

## New Model 2: Empirical Semi-Markov Timeout Model

This model does not assume exponential waiting times. It uses observed outgoing transition samples and treats waits over 60 days as failure. It estimates the recursive probability of eventually reaching success before a timeout.

In [ ]:
semi_markov = SemiMarkovTimeoutModel().fit(timeout_transitions)
semi_probs_by_state = pd.DataFrame({
    "state": semi_markov.states_,
    "p_success_before_timeout_failure": semi_markov.predict_success_probability(semi_markov.states_),
}).sort_values("p_success_before_timeout_failure", ascending=False)
display(semi_probs_by_state)

## Diagnostic Comparison on Resolved Journeys

This joins model scores to resolved labels. It is diagnostic rather than a perfect offline validation because these features are built from completed journeys. For Kaggle-style evaluation, prefer open/test-prefix submissions.

In [ ]:
labels = data.load_binary_labels()
eval_df = features.merge(labels, on="id", how="inner")

predictions = {
    "global_ctmc_old_target": global_ctmc.absorption_probability(eval_df["current_state"]),
    "clustered_ctmc_old_target": clustered.predict_success_probability(eval_df, fallback_model=global_ctmc),
    "timeout_absorbing_ctmc": timeout_ctmc.predict_success_probability(eval_df["current_state"]),
    "semi_markov_timeout": semi_markov.predict_success_probability(eval_df["current_state"]),
}

y = eval_df["label"].astype(int).to_numpy()
rows = []
for name, probs in predictions.items():
    probs = pd.Series(probs).clip(1e-6, 1 - 1e-6).to_numpy()
    rows.append({
        "model": name,
        "brier_score": brier_score_loss(y, probs),
        "log_loss": log_loss(y, probs, labels=[0, 1]),
        "roc_auc": roc_auc_score(y, probs),
        "average_precision": average_precision_score(y, probs),
        "mean_prob": probs.mean(),
    })

comparison = pd.DataFrame(rows).sort_values("brier_score")
comparison.to_csv(RESULTS_DIR / "ctmc_timeout_model_comparison.csv", index=False)
display(comparison)

## Per-State Probability Comparison

In [ ]:
all_states = sorted(set(global_ctmc.states_) | set(timeout_ctmc.states_) | set(semi_markov.states_))
all_states = [s for s in all_states if s not in {SUCCESS_STATE, FAILURE_STATE}]
state_comparison = pd.DataFrame({
    "state": all_states,
    "global_hit_success_60d": global_ctmc.absorption_probability(all_states),
    "timeout_absorbing_success_before_failure": timeout_ctmc.predict_success_probability(all_states),
    "semi_markov_success_before_failure": semi_markov.predict_success_probability(all_states),
})
display(state_comparison.sort_values("timeout_absorbing_success_before_failure", ascending=False))

state_comparison.set_index("state").plot.bar(figsize=(12, 5))
plt.ylabel("probability")
plt.title("Old success-hit target vs timeout-aware target")
plt.tight_layout()
plt.show()

## Submission Commands

From PowerShell, run the full retained CTMC set:

```powershell
python -m src.models.ctmc_pipeline all --no-cache
```

Run only spectral clustered CTMC:

```powershell
python -m src.models.ctmc_pipeline clustered --use-spectral-clustering --no-cache
```